In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# INPUT / OUTPUT
INPUT_CSV = Path("data/MASTER_VARIABLES.csv")
OUTPUT_CSV = Path("data/hazard.csv")

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# 1. DISTRICT-MONTH MEAN
# ---------------------------------------------------------
district_stats = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(district_mean_heatday=("mean_heatday", "mean"))
)

# ---------------------------------------------------------
# 2. MONTHLY Z-SCORE
# ---------------------------------------------------------
district_stats["heat_zscore"] = (
    district_stats.groupby("timeperiod")["district_mean_heatday"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) != 0 else 0)
)

# ---------------------------------------------------------
# 3. BINNING (1–5)
# ---------------------------------------------------------
def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

district_stats["heat_hazard"] = district_stats["heat_zscore"].apply(classify)

# ---------------------------------------------------------
# 4. SAVE ONLY DISTRICT-MONTH OUTPUT (CLEAN)
# ---------------------------------------------------------
district_stats.to_csv(OUTPUT_CSV, index=False)

print(f"Saved clean hazard file: {OUTPUT_CSV}")
print(district_stats.head())

Saved clean hazard file: data/hazard.csv
  district timeperiod  district_mean_heatday  heat_zscore  heat_hazard
0   Anugul    2023_01              11.563157    -0.021585            3
1   Anugul    2023_02              11.563157    -0.021585            3
2   Anugul    2023_03              11.563157    -0.021585            3
3   Anugul    2023_04              32.145668    -0.099194            3
4   Anugul    2023_05              32.145668    -0.099194            3
